<a href="https://colab.research.google.com/github/kgeorge-fission/Resources/blob/feat-prompt-evaluation/examples/flotorch-evaluation-notebooks/prompt-evaluation/prompt_evaluation.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Prompt Evaluation Notebook

This notebook evaluates multiple **system-user prompt and model pairs** for **question answering tasks** using the **Flotorch SDK** and **Flotorch Eval**.

## What it Does

This notebook runs structured benchmarking and prompt experimentation to identify the most effective prompting strategies for your LLM-based RAG or QA system. It supports the following capabilities:

### Core Evaluation Features
- Performs automated evaluation on LLM responses using retrieved context.
- Computes the following quality metrics:
  - **Context Precision** - Measures how focused the retrieved context is to the query.
  - **Faithfulness** - Measures factual consistency of the answer with the given context.
  - **Answer Relevancy** - Measures how well the answer addresses the question.
  - **Context Relevancy** - Evaluates the relevance of retrieved context to the question.
  - **Context Recall** - Measures how much of the necessary ground-truth information is present in the context.
  - **Hallucination** - Detects whether the answer contains unsupported or incorrect claims.
  - **Maliciousness** - Detects if the response contains harmful or unsafe content.

### Performance, Cost, and Ranking
- Calculates:
  - **Average metric score** per prompt pair
  - **Average response latency**
  - **Total tokens used**
  - **Total cost (USD)**
- Selects the **best-performing prompt pair** using a **weight-based scoring system**:
  - Assign weights to **Quality Score**, **Cost**, and **Latency**
  - The notebook identifies the best prompt pair based on your performance priorities

### Additional Advanced Capabilities
This notebook also supports several prompt engineering and experimentation features:

| Feature | Description |
|--------|-------------|
| **Models and Knowledge Base Validation** | Validates availability and configuration of all LLM, embedding models, and knowledge bases before running experiments |
| **Automatic Prompt Generation** | Automatically expands your prompt set by generating **N new prompt variations** using an LLM |
| **Ground-Truth Question Rephrasing** | Rephrases each evaluation question using an LLM to potentially produce higher quality responses |
| **Context Experimentation** | Lets you experiment with different numbers of retrieved context items to determine the optimal context size |
| **Prompt Assembly** | Allows customizable LLM payload construction using JSON to define message structure (system, assistant, user roles, etc.) |

## Output

The notebook outputs:
- A list of evaluated prompt pairs with full metric breakdown
- Aggregate latency, cost, and token usage
- Rank-ordered list of prompt pairs based on your selected weight configuration
- The **top-performing prompt pair** and its details

## Requirements to Run the Notebook

You will need:
- A **Flotorch account** with LLM and embedding models configured
- A **`gt.json`** file with ground truth QA pairs (format shown later in the notebook)
- A **`prompts.json`** file containing system-user prompt pairs (format shown later)
- (Optional) An LLM configured for automatic prompt generation and question rephrasing

---

You can now run structured evaluations, cost-latency-quality tradeoff comparisons, automatic prompt engineering, and context experimentation all from one unified workflow.


In [1]:
# Install flotorch-sdk and flotorch-core
# You can safely ignore the dependency errors during the installation.

!pip install flotorch==2.2.0b1 flotorch-eval==1.2.0b1 -q

### Model Setup  

Before running the experiment, you first need to configure your Flotorch environment so the notebook can communicate with your console and access the models you've created.  

Set the following variables:  
- **`FLOTORCH_API_KEY`** - your Flotorch API key.  
- **`FLOTORCH_BASE_URL`** - the base URL of your Flotorch console instance.  

---

Before running the experiment, define the core models that will be used throughout this notebook.  

#### 1. **`inferencer_model_names`**  
This is a list of **generative LLM model** used to produce answers for each question in the ground truth (GT) data.  
You can configure these model in the **Flotorch Console** by first adding a provider and then creating a model under that provider. Each model will be run across all the system-user prompt pairs and questions.

#### 2. **`evaluation_llm_model_name`**  
This is the **evaluation LLM model** used to assess the quality of the responses generated by the inferencer model.  
Similar to the inferencer model, it should also be created in the Flotorch Console under a provider of your choice.  

#### 3. **`evaluation_embedding_model_name`**  
This is the **embedding model** used during evaluation to measure semantic similarity between responses and reference answers.  
To set this up in the console, create a provider and append the model name to it.  

**Example:**  
If you create a provider named `openai-provider` and want to use the `text-embedding-ada-002` embedding model, then your `evaluation_embedding_model_name` will be:  

openai-provider/text-embedding-ada-002

#### **Best Practice:**
It’s generally recommended to keep your inferencer and evaluation models separate.
Using different models helps ensure that the evaluation remains unbiased — if the same model that generated the responses also judges them, it may overestimate its own performance. Separate models provide a more objective and reliable assessment.

In [2]:
import getpass

try:
  FLOTORCH_API_KEY=getpass.getpass("Paste your API key here: ")
  print(f"Success")
except getpass.GetPassWarning as e:
    print(f"Warning: {e}")

FLOTORCH_BASE_URL=input("Paste your Flotorch Base URL here: ")

Success


In [ ]:
# Flotorch LLM model for inferencing
inference_model_names = [""]

# Flotorch LLM model for evaluation
evaluation_llm_model_name = ""

# Flotorch embedding model for evaluation
# use the format <model_provider>/<model_name> where <model_provider> is the provider you've set up in Flotorch Console and <model_name> is the actual model name.
# Eg: openai-provider/text-embedding-ada-002
evaluation_embedding_model_name = "<openai-provider>/text-embedding-ada-002"


# This is the knowledge base repository you have set up on Flotorch console.
# This is optional and can be left as an empty string or commented out if you don't have a knowledge base.
# Note: Some metrics that depend on the cotext like 'context relevancy' will not be used if knowledge base is not passed.
knowledge_base_repo = ""

### Load Project Code

This step retrieves the project source code from the repository and makes the `prompt_evaluation` module available for import.  
All core components—including prompt generation, evaluation logic, scoring utilities, and workflow orchestration—are defined in this module, so it must be loaded before running the evaluation pipeline.


In [ ]:
!git clone --branch feat-prompt-evaluation https://github.com/kgeorge-fission/Resources.git
import sys
sys.path.append("/content/Resources/examples/flotorch-evaluation-notebooks/prompt-evaluation")

### Imports  

The following libraries and modules are required for this notebook.  
They include flotorch eval imports and other core modules with utilities designed for this notebook.


In [ ]:
#Required imports
import json

from flotorch_eval.llm_eval import LLMEvaluator, MetricKey

from core import (
    ExperimentRunner,
    validate_environment,
    generate_prompts,
    EvaluationDatasetType,
    PromptEvaluationResult,
    display_prompt_results,
    best_prompt_pair,
    get_weighted_scores
)


### Upload Ground Truth and Prompts Files  

Use the file upload widgets below to upload your **Ground Truth (gt.json)** and **Prompts (prompts.json)** files.  
The notebook will automatically read and load them into their variables which will be used in later cells.

Context can be provided either through the ground truth or as a Flotorch knowledge base.

#### **Expected File Formats**

**Ground Truth (`gt.json`)**
```json
[
  {
    "question": "What is Amazon Bedrock?",
    "answer": "Amazon Bedrock is a fully managed service that makes foundation models available through an API.",
    "context": [<item1>, <item2>...]
  },
    {
    "question": "Which FMs are available on Amazon Bedrock?",
    "answer": "Amazon Bedrock customers can choose from some of the most cutting-edge FMs available today. Currently we offer 47 models.",
    "context": [<item1>, <item2>...]
  },
  ...
]
```

**Prompt pairs (`prompts.json`)**
```json
[
  {
    "system_prompt": "You are an AI assistant that provides accurate answers based on the given context.",
    "user_prompt": "Answer the following question using the provided context."
  },
  {
    "system_prompt": "You are an AWS cloud documentation assistant trained on Amazon Bedrock materials. Read the provided context carefully and answer only from it.",
    "user_prompt": "Using only the retrieved Bedrock context, give a short and factual answer to the question below."
  },
  ...
]
```

You can either:
- Upload your files to your Google drive and load the files here by providing the path: This is better if you have multiple files and plan to switch them frequently.

OR

- You can upload the files directly in the cell below and those will be stored in the Google Colab temporary storage for the experimentation: This is good if you only plan to run a file once.

In [ ]:
# If you don't have files in your local directory, you can use the following code to upload them.

# Uncomment the below code if you're planning to do this

from google.colab import files
print("Please upload your Ground Truth file (gt.json)")
gt_upload = files.upload()

gt_path = list(gt_upload.keys())[0]
with open(gt_path, 'r') as f:
    ground_truth = json.load(f)
print(f"Ground truth loaded successfully — {len(ground_truth)} items\n")


print("Please upload your Prompts file (prompts.json)")
prompts_upload = files.upload()

prompts_path = list(prompts_upload.keys())[0]
with open(prompts_path, 'r') as f:
    prompts = json.load(f)
print(f"Prompts loaded successfully — {len(prompts)} prompt pairs")

In [ ]:
# If you already have files in your local directory, you can use the following code to load them.

# Uncomment the below code if you're planning to do this

# GT_PATH = "data/100_combined_gt_retrieval_ada002.json"
# with open(GT_PATH, 'r') as f:
#     ground_truth = json.load(f)
# print(f"Ground truth loaded successfully — {len(ground_truth)} items\n")

# PROMPT_PATH = "data/prompts_test.json"
# with open(PROMPT_PATH, 'r') as f:
#     prompts = json.load(f)
# print(f"Prompt loaded successfully.")

### OPTIONAL: Convert ground truth format to the format required by the ExperimentRunner

The below function converts ground-truth data that contains **object-based context items** into a **string-only context list**, which is the format required by `ExperimentRunner`.

#### **Purpose**

Some ground-truth datasets store context as:

- A list of objects like  
  ```json
  { "chunk_text": "..." }
  ```

ExperimentRunner expects the context to be a list of strings, so this function normalizes every possible input format into that canonical structure

- Context expected by ExperimentRunner
```json
"context": [
            "AI stands for artificial intelligence.",
            "It is a field of computer science."
        ]
```

**Note**: If your data contains context in that format, use the below function to convert it. Feel free to modify the function further if it does not match the type of data you have.

In [7]:
from core import convert_ground_truth_format

ground_truth = convert_ground_truth_format(ground_truth)

In [ ]:
ground_truth

### Optional: Rephrase Questions

This optional step allows you to rephrase questions in your ground truth data to improve their effectiveness before running experiments.

#### **Why Rephrase Questions?**

- **Improve Question Quality**: Transform naive or overly simple questions into more sophisticated, nuanced queries that can elicit better answers from LLMs
- **Reduce Token Usage**: Condense verbose questions into more concise versions while preserving meaning, potentially reducing input token costs
- **Better Context Matching**: More precise questions can lead to better context retrieval and more relevant answers
- **Maintain Context**: The rephrasing preserves the exact same context and intent - only the phrasing changes

#### **How It Works**

The `rephrase_questions()` function uses an LLM to intelligently rephrase each question in your ground truth data. The function:
- Analyzes each question to identify areas for improvement
- Makes questions more sophisticated if they're too simple
- Reduces verbosity if questions are overly wordy
- Preserves all key concepts and maintains the same semantic meaning

#### **Usage**

Simply run the cell below to rephrase your questions. The function will:
1. Process each question in your ground truth data
2. Use the specified LLM model to generate improved versions
3. Return a new ground truth dataset with rephrased questions
4. Optionally preserve all original answers

**Note:** This step is completely optional. You can skip it if your questions are already well-formulated.


In [ ]:
from core import rephrase_questions

# Set the model that you want to use for rephrasing
rephrasing_llm = "flotorch/gpt-oss-120b"

ground_truth = rephrase_questions(
    ground_truth=ground_truth,
    llm=rephrasing_llm,
    api_key=FLOTORCH_API_KEY,
    base_url=FLOTORCH_BASE_URL,
    preserve_original=True  # Set to True if you want to keep original questions
)
# If you preserve the original, it will be added as another value in the ground
# truth with a key 'original_question'.

# After rephrasing, you can inspect the results:
for i, item in enumerate(ground_truth):
    print(f"\nQuestion {i+1}:")
    if "original_question" in item:
        print(f"  Original: {item['original_question']}")
        print(f"  Rephrased: {item['question']}")
    else:
        print(f"  {item['question']}")


In [ ]:
# Inspect the new ground truth
ground_truth

### OPTIONAL: Prompt Generation System

This feature provides an automated way to generate improved **system + user prompt pairs** for Retrieval-Augmented Generation (RAG) tasks. It is designed to help optimize prompts for better grounding, relevance, and factual consistency.

---

## What This Feature Does

The prompt generation system creates new prompt pairs that are:

- Distinct from previously used prompts  
- Tailored to improve RAG metrics such as Context Precision, Faithfulness, Answer Relevancy, Context Recall, and Hallucination reduction  
- Designed using an identifiable strategy (e.g., evidence citation, context validation, structured reasoning)

Each generated prompt includes:
- A refined *system prompt*
- A refined *user prompt*
- The strategy used to design it
- An explanation of the expected improvement

---

## Why Use It

- It systematically explores different prompting strategies without manual effort  
- It helps improve the quality and reliability of RAG-generated answers  
- It encourages better context use and reduces hallucinations  
- Each iteration uses all previous prompts to avoid repetition and promote meaningful variation  

This makes it valuable when tuning, comparing, or evolving prompts for production or evaluation.

---

## How It Works

The system takes existing prompt pairs and feeds them, along with explicit instructions, into an LLM. The model analyzes the structure, identifies weaknesses, and generates a new prompt pair that differs in style or strategy. Each iteration:

1. Collects all existing prompts  
2. Builds a generation request using predefined system and user prompt templates  
3. Invokes the LLM with a strict JSON response format  
4. Parses and stores the new prompt  
5. Adds it to the pool so future iterations produce varied results  

The main function, `generate_prompts()`, runs this loop for `n` iterations and returns a list of newly generated prompts.

---

## Key Components

### Prompt Templates  
- `PROMPT_GENERATION_SYSTEM` defines the rules and goals for creating improved prompts.  
- `PROMPT_GENERATION_USER` supplies existing prompts and the JSON schema for output.  

### Schema Model  
`GeneratedPrompt` defines the expected fields for each generated prompt pair.

### Parsing  
`parse_generated_prompts()` ensures robust processing of JSON responses.

### Generation Logic  
`generate_prompts()` orchestrates environment validation, LLM invocation, schema enforcement, and prompt iteration.

---

This module provides a structured, repeatable method for evolving and improving prompt design for RAG systems.


In [ ]:
prompt_generation_llm = ""

new_prompts = generate_prompts(
    llm=prompt_generation_llm,
    prompts=prompts,
    n=3,
    api_key=FLOTORCH_API_KEY,
    base_url=FLOTORCH_BASE_URL
)

In [13]:
# Inspect the new generated prompts
new_prompts

[{'system_prompt': 'You are a retrieval‑augmented generation specialist for Amazon Bedrock documentation. Given the extracted PDF excerpts, produce an answer that (1) cites the exact sentence(s) from the context supporting each claim using brackets, (2) limits the response to the information present, and (3) if no supporting text exists, replies with the mandated phrase. Maintain a neutral tone and avoid any speculation.',
  'user_prompt': "Using only the provided Bedrock excerpts, answer the question below. For every factual statement, include a citation in the form [source] where 'source' is the exact snippet from the context. Keep the answer concise and do not add information beyond what is cited. If the answer cannot be found, respond with 'The information is not available in the provided context.'"},
 {'system_prompt': "You are a Retrieval‑Augmented Generation specialist for Amazon Bedrock documentation. Your workflow must be:\n1. Scan the supplied context and locate every sentenc

In [ ]:
# If you are satisfied with the generated prompts, you can extend the prompts
# with new ones by running the following cell
prompts.extend(new_prompts)

# Inspect the complete prompts
prompts


## Run Experiments  

This section defines and executes the asynchronous experiment workflow.  

It initializes the asynchronous functions responsible for running experiments and performing knowledge base searches. The process iterates through each **model** in the list, through each **system-user prompt pair**, runs it against all question-answer entries in the ground truth dataset, and compiles the results into a structured dictionary. This continues until every prompt pair has been evaluated.  

**Note:** This is a compute-intensive process since each model is executed for every question and prompt pair.  
For example, with **3 models**, **10 prompt pairs**, **2 context sets** and **10 questions**, the total number of LLM queries will be **3 x 10 x 10 = 600**.


### Assembly Rule

The **assembly rule** defines how different components — such as the system prompt, user prompt, context, n-shot examples, and questions — are combined to form the final input for the model. The way these pieces are stitched together significantly impacts the quality of the model’s responses.

This notebook allows users to **easily customize** the assembly structure through a JSON object.

#### **Default Assembly Structure**
```python
assembly_rule = {
    "separator": "",
    "system_prompt": ["system", "context", "examples"],
    "user_prompt": ["user", "question"]
}
```
#### How It Works

Everything listed under system_prompt is concatenated (using the defined separator) and passed to the model as the system role.

Everything listed under user_prompt is concatenated (using the same separator) and passed to the model as the user role.

Available Components

You can use the following values within either system_prompt or user_prompt:

system → The system prompt provided by the user.

user → The user prompt provided by the user.

context → The retrieved or provided contextual information.

examples → The n-shot examples included alongside the prompts.

question → Each question present in the ground truth (GT) data.

Separator

The "separator" key defines what string (e.g., "\n\n", "---", " ") is used to join multiple components.

You can set it to any value depending on how you want the content to be structured in the final payload.

This flexible rule-based approach makes it easy to experiment with different prompt assemblies and evaluate their impact on model performance.

In [14]:
assembly_rule={
        "separator": "\n",
        "system_prompt": ["system"],
        "user_prompt": ["user", "context", "examples", "question"]
    }


### Context Experimentation

The **context experimentation feature** allows you to systematically test how different context retrieval strategies and context sizes impact model performance. This enables you to find the optimal balance between response quality, latency, and cost.

#### **What is Context Experimentation?**

When using a knowledge base, the system retrieves relevant context chunks for each question. Context experimentation lets you:
- **Test multiple context sizes** — Compare performance with different numbers of retrieved context chunks (e.g., 1, 3, 5 chunks)
- **Experiment with retrieval strategies** — Choose how context chunks are selected:
  - **`"top"`** — Uses the most relevant chunks based on similarity scores
  - **`"random"`** — Randomly selects chunks (useful for testing robustness)

#### **Why It Matters**

Different context sizes can significantly affect:
- **Answer Quality** — More context may provide better answers, but can also introduce noise
- **Latency** — Larger context sizes increase processing time
- **Cost** — More context means more tokens, increasing API costs
- **Faithfulness** — The right amount of context helps models stay grounded in retrieved information

#### **How It Works**

The `ExperimentRunner` automatically runs experiments for each combination of:
- Model × Prompt Pair × Question × Context Size

For example, with **1 model**, **5 prompt pairs**, **10 questions**, and **2 context sizes** `[1, 3]`:
- Total experiments = 1 × 5 × 10 × 2 = **100 LLM calls**
- Results are grouped by context size for easy comparison

#### **Configuration**

Set the following parameters when initializing `ExperimentRunner`:
- **`context_sizes`** — List of context chunk counts to test (e.g., `[1, 3, 5]`)
- **`context_strategy`** — How to select chunks: `"top"` or `"random"`
- **`random_seed`** — Seed for reproducible random selection (when using `"random"` strategy)

The evaluation results will include separate metrics for each context size, allowing you to identify the optimal configuration for your use case.

#### **Automatic Knowledge Base Retrieval**

If you provide context in both ground truth and knowledge base, the context in the ground truth will take precedence. Only if the context is not available in the ground truth will it use the knowledge base.

When using a knowledge base with `context_sizes`, the system **automatically optimizes retrieval** to ensure you get enough chunks for all your experiments:

- **Automatic Calculation**: The system calculates the maximum value from your `context_sizes` list (e.g., if `context_sizes=[2, 10]`, it uses `max_results=10`)
- **Efficient Retrieval**: Instead of making separate knowledge base queries for each context size, it retrieves **all chunks needed** in a single query using the `max_number_of_result` parameter
- **Smart Filtering**: After retrieval, the system filters the chunks to the specific sizes needed (e.g., top 2 chunks or top 10 chunks) based on your `context_strategy`

**Example:**
If you specify `context_sizes=[1, 3, 5]`:
- The system retrieves up to **5 chunks** from the knowledge base (the maximum)
- Then filters them to 1, 3, or 5 chunks as needed for each experiment
- This avoids multiple knowledge base queries and ensures you have enough chunks for all context sizes

**Note:** If you specify a context size larger than what's available in the knowledge base, the system will log a warning and skip that context size for that question, but continue with other experiments.



#### **Configuration**

The `ExperimentRunner` class accepts the following parameters:

**Required Parameters:**
- **`models`** (`List[str]`) — List of LLM model names to test. Each model will be evaluated across all prompt pairs and questions. Models should be configured in your Flotorch Console (e.g., `["flotorch/haiku-long"]`).
- **`prompts`** (`List[Dict[str, Any]]`) — List of prompt dictionaries, each containing `system_prompt` and `user_prompt` keys. Optionally can include `examples` for n-shot learning.
- **`ground_truth`** (`List[Dict[str, Any]]`) — List of question-answer pairs. Each dictionary should contain `question` and `answer` keys. Optionally can include `context` for pre-provided context.
- **`api_key`** (`str`) — Your Flotorch API key for authentication.
- **`base_url`** (`str`) — The base URL of your Flotorch console instance.

**Optional Parameters:**
- **`knowledge_base`** (`Optional[str]`) — Name of the knowledge base repository configured in Flotorch Console. If provided, context will be retrieved from this knowledge base. If `None`, context must be provided in the ground truth data. Default: `None`.
- **`context_sizes`** (`Optional[List[int]]`) — List of context chunk counts to test (e.g., `[1, 3, 5]`). Each size will be tested for all model-prompt-question combinations. If `None`, all available context chunks will be used. Default: `None`.
- **`context_strategy`** (`str`) — Strategy for selecting context chunks when `context_sizes` is specified:
  - `"top"` — Uses the most relevant chunks based on similarity scores (default)
  - `"random"` — Randomly selects chunks (useful for testing robustness)
  Default: `"top"`.
- **`random_seed`** (`Optional[int]`) — Seed for reproducible random selection. Used when `context_strategy="random"` or when sampling n-shot examples. If set, ensures reproducible results across runs. Default: `None`.
- **`assembly_rule`** (`Optional[Dict]`) — Custom rule for assembling prompt components. Defines how system prompt, user prompt, context, examples, and question are combined. See the "Assembly Rule" section above for details. Default: `None` (uses default assembly).
- **`n`** (`Optional[int]`) — Number of n-shot examples to use from each prompt's examples list. If specified, randomly samples `n` examples from available examples. If fewer than `n` examples are available, uses all available examples and logs a warning. If `None`, uses all available examples. Default: `None`.

**Example:**
```python
runner = ExperimentRunner(
    models=["flotorch/haiku-long"],
    prompts=prompts,
    ground_truth=ground_truth,
    api_key=FLOTORCH_API_KEY,
    base_url=FLOTORCH_BASE_URL,
    knowledge_base="bedrock-kb",
    context_sizes=[1, 3],
    context_strategy="top",
    random_seed=42,
    assembly_rule=assembly_rule,
    n=3  # Use 3 randomly sampled examples per prompt
)
```

The evaluation results will include separate metrics for each context size, allowing you to identify the optimal configuration for your use case.

In [ ]:
runner = ExperimentRunner(
    models=inference_model_names,
    prompts=prompts,
    ground_truth=ground_truth,
    api_key=FLOTORCH_API_KEY,
    base_url=FLOTORCH_BASE_URL,
    knowledge_base=knowledge_base_repo,
    context_sizes=[3],
    context_strategy="top",   # or "random"
    random_seed=42,
    assembly_rule=assembly_rule,
    n=1
)

evaluation_data = await runner.run_async(concurrency=10)

In [16]:
evaluation_data

{'units_count': 3,
 'warnings': [],
 'runs': [{'model': 'flotorch/haiku-lon',
   'system_prompt': "You are an expert AI researcher specializing in cloud computing and AWS services, particularly Amazon Bedrock. Your task is to extract information from the provided Amazon Bedrock PDF and generate precise, structured, and concise answers to user questions. Ensure each response is specific, directly relevant to the question, and clearly reflects the content of the document. If the information is not found in the provided context, respond only with 'The information is not available in the provided context.'",
   'user_prompt': 'Based on the above retrieved context, answer the following question clearly and concisely, avoiding repetition.',
   'context_size': 3,
   'experiments': [EvaluationItem(question='Which clinically actionable alterations have been reported in the CALR gene?', generated_answer='', expected_answer='The CALR gene has oncogenic mutations that are considered actionable, pa

## Evaluate Results  

This section initializes the models required for evaluation using the model names you have provided and computes performance metrics for each system-user prompt pair.

The `run_evaluation()` function:  
- Initializes Flotorch LLMEvaluator using your inference llm and embedding model
- By default LLMEvaluator runs on all the metrics that is available.
Available metrics are:
    - Context precision
    - Context recall
    - Context relevancy
    - Answer relevancy
    - Hallucination
    - Faithfulness
    - Aspect critic
- Aspect critic requires a definition of the aspect and that is provided to the LLMEvaluator in the metric config.
- Providing the LLMEvaluator with headers activates gateway metrics:
    - Total tokens
    - Total cost
    - average and total latency
- Computes an overall **average score** for each prompt pair and returns a structured summary of all evaluations.

### **What does these metrics mean?**

**Context Precision:**  
Measures how well the retrieved context supports the generated answer. High context precision indicates that the model effectively retrieves focused and useful information rather than unrelated or noisy context.

**Context recall**
Measures how much of the relevant information from the source context was used in the generated answer. High context recall means the model successfully captured most of the necessary details from the retrieved context.

**Context Relevancy**
Evaluates how relevant the retrieved context is to the user's question. High context relevancy indicates that the retrieved passages are strongly aligned with what the question is asking.

**Answer Relevancy:**  
Evaluates how relevant and appropriate the model's answer is to the user's question.

**Hallucination**
Assesses whether the model introduces information not present or supported by the provided context. A lower hallucination score indicates that the model stays faithful to the retrieved data without adding fabricated or unrelated details.

**Faithfulness:**  
Assesses whether the generated answer is factually consistent with the retrieved context.

**Maliciousness:**  
An aspect-critic metric that determines whether the model's response contains harmful, toxic, or unsafe content. It assesses the presence of language or reasoning that could be considered offensive, dangerous, or unethical in nature.

*Note:* You may see some warnings or errors printed to the console during execution. These come from the RAGAS library and can be safely ignored — they do not affect the evaluation flow.


In [17]:
def run_evaluation(evaluation_data: EvaluationDatasetType) -> PromptEvaluationResult:
    import math

    data_runs = evaluation_data.get("runs", evaluation_data)

    metric_args = {
        MetricKey.ASPECT_CRITIC: {
            "maliciousness": {
                "name": "maliciousness",
                "definition": "Is the response harmful or toxic?"
            }
        }
    }

    # Validate models
    validate_environment(
        llm_models=[evaluation_llm_model_name],
        embedding_models=[evaluation_embedding_model_name],
        api_key=FLOTORCH_API_KEY,
        base_url=FLOTORCH_BASE_URL
    )

    client = LLMEvaluator(
        api_key=FLOTORCH_API_KEY,
        base_url=FLOTORCH_BASE_URL,
        embedding_model=evaluation_embedding_model_name,
        inferencer_model=evaluation_llm_model_name,
        metrics=[MetricKey.ANSWER_RELEVANCE, MetricKey.ASPECT_CRITIC],
        evaluation_engine='ragas',
        metric_configs=metric_args
    )

    results = []

    for prompt_set in data_runs:
        try:
            eval_result = client.evaluate(prompt_set.get("experiments"))
            eval_metrics = eval_result.get("evaluation_metrics", {})
            gateway_metrics = eval_result.get("gateway_metrics", {})

            numeric_values = [
                v for v in eval_metrics.values()
                if isinstance(v, (int, float))
                and not (isinstance(v, float) and (math.isnan(v) or math.isinf(v)))
            ]

            if numeric_values:
                average_score = sum(numeric_values) / len(numeric_values)
                eval_metrics['average_score'] = round(average_score, 2)
            else:
                eval_metrics['average_score'] = 0.0

            if gateway_metrics:
                eval_metrics.update(gateway_metrics)

            results.append(
                {
                    "inference_model": prompt_set.get("model"),
                    "system_prompt": prompt_set.get("system_prompt"),
                    "user_prompt": prompt_set.get("user_prompt"),
                    "context_size": prompt_set.get("context_size"),
                    "evaluation_metrics": eval_metrics
                }
            )

        except Exception as e:
            print(f"Error evaluating prompt set: {e}")
            results.append(
                {
                    "inference_model": prompt_set.get("model"),
                    "system_prompt": prompt_set.get("system_prompt"),
                    "user_prompt": prompt_set.get("user_prompt"),
                    "context_size": prompt_set.get("context_size"),
                    "evaluation_metrics": {
                        "average_score": 0.0,
                        "error": str(e)
                    }
                }
            )

    return results


In [ ]:
results: PromptEvaluationResult = run_evaluation(evaluation_data)

In [19]:
results

[{'inference_model': 'flotorch/haiku-lon',
  'system_prompt': "You are an expert AI researcher specializing in cloud computing and AWS services, particularly Amazon Bedrock. Your task is to extract information from the provided Amazon Bedrock PDF and generate precise, structured, and concise answers to user questions. Ensure each response is specific, directly relevant to the question, and clearly reflects the content of the document. If the information is not found in the provided context, respond only with 'The information is not available in the provided context.'",
  'user_prompt': 'Based on the above retrieved context, answer the following question clearly and concisely, avoiding repetition.',
  'context_size': 3,
  'evaluation_metrics': {'answer_relevancy': 0.0,
   'maliciousness': 1.0,
   'average_score': 0.5}},
 {'inference_model': 'flotorch/haiku-lon',
  'system_prompt': "You are an AWS cloud documentation assistant trained on Amazon Bedrock materials. Read the provided contex

### Weighted Scoring Configuration

This section allows you to **assign weights** to different evaluation dimensions —  
**response quality**, **latency**, and **cost** — based on what matters most for your use case.

The notebook will then compute a new **`weighted_final_score`** for each prompt-model pair using these weights.

---

#### **How Weighted Scoring Works**

Each evaluated prompt pair produces:
- `average_score` → Quality of the answer (**higher is better**)  
- `average_latency_ms` → Speed of generation (**lower is better**)  
- `average_cost` → Cost in USD (**lower is better**)

Since latency and cost are *better when lower*, they are automatically **normalized and inverted** during weighting so that all values align directionally (**higher = better**).

---

#### **Set Your Custom Weights**

Adjust the following values based on your priorities:  

| Metric | Description | Suggested Range | Example |
|:--|:--|:--:|:--:|
| `average_score` | Importance of response quality | 0.0-1.0 | `0.6` |
| `average_latency_ms` | Importance of response speed | 0.0-1.0 | `0.2` |
| `average_cost` | Importance of cost efficiency | 0.0-1.0 | `0.2` |

**Note:**  
- The notebook will automatically normalize your weights so they sum to 1.  
- If you skip this section, default weights of **(0.6, 0.2, 0.2)** will be used.

---


In [20]:
user_weights = {
    "average_score": 0.4,
    "average_latency_ms": 0.3,
    "average_cost": 0.3
}
results = get_weighted_scores(results, user_weights)

In [21]:
best_prompt_pair(results)


 Best Performing Model-Prompt Combination:
   • Model: flotorch/haiku-lon
   • Context Size: 3
   • Average Score: 0.5
   • Faithfulness: N/A
   • Answer Relevance: 0.0
   • Context Precision: N/A
   • Contextual Relevancy: N/A
   • Contextual Recall: N/A
   • Hallucination: N/A
   • Maliciousness: 1.0
   • Avg Latency (ms): N/A
   • Total Latency (ms): N/A
   • Total Tokens: N/A
   • Total Cost (USD): N/A

   • System Prompt:
You are an expert AI researcher specializing in cloud computing and AWS
services, particularly Amazon Bedrock. Your task is to extract information from
the provided Amazon Bedrock PDF and generate precise, structured, and concise
answers to user questions. Ensure each response is specific, directly relevant
to the question, and clearly reflects the content of the document. If the
information is not found in the provided context, respond only with 'The
information is not available in the provided context.'

   • User Prompt:
Based on the above retrieved context, 

In [22]:
results

[{'inference_model': 'flotorch/haiku-lon',
  'system_prompt': "You are an expert AI researcher specializing in cloud computing and AWS services, particularly Amazon Bedrock. Your task is to extract information from the provided Amazon Bedrock PDF and generate precise, structured, and concise answers to user questions. Ensure each response is specific, directly relevant to the question, and clearly reflects the content of the document. If the information is not found in the provided context, respond only with 'The information is not available in the provided context.'",
  'user_prompt': 'Based on the above retrieved context, answer the following question clearly and concisely, avoiding repetition.',
  'context_size': 3,
  'evaluation_metrics': {'answer_relevancy': 0.0,
   'maliciousness': 1.0,
   'average_score': 0.5,
   'weighted_final_score': 1.0}},
 {'inference_model': 'flotorch/haiku-lon',
  'system_prompt': "You are an AWS cloud documentation assistant trained on Amazon Bedrock mat

In [23]:
display_prompt_results(results)


EXPERIMENT SUMMARY

Total Number of experiments: 3
Models tested: 1
Prompts tested: 3
Context sizes tested: [3]

TOP 5 PERFORMING CONFIGURATIONS
(Automatically uses weighted score if available)
+--------+--------------------+----------------+-------------+------------------+--------------+----------------+----------------+--------------+----------------------------------------------------+----------------------------------------------------+
|   Rank | Model              |   Context Size |   Avg Score |   Weighted Score |   Answer Rel |   Faithfulness |   Latency (ms) |   Cost (USD) | System Prompt                                      | User Prompt                                        |
+========+====================+================+=============+==================+==============+================+================+==============+====================================================+====================================================+
|      1 | flotorch/haiku-lon |              3 | 